In [0]:
import os
import numpy as np
import pandas as pd
import random
from pickle import dump
from keras.callbacks import ModelCheckpoint
from keras.models import Sequential
from keras.layers import LSTM, Dropout, TimeDistributed, Dense, Activation, Embedding
from keras.utils import to_categorical

In [0]:
SEQ_LEN = 64
#length of each sequence used to predict next character

In [0]:
file = open('complete_data_tunes.txt', mode='r')
#file contains lyrics of songs in abc format
data = file.read()
file.close()

In [4]:
print(len(data))
#total number of characters

175731


**DATA PREPROCESSING**

In [0]:
char_to_index = {ch : i for (i,ch) in enumerate(sorted(list(set(data))))}
#dictionary to store index values for each character

In [0]:
index_to_char = {i : ch for (ch,i) in char_to_index.items()}
#reverse dictionary to store characters for each index

In [7]:
unique_chars = len(char_to_index)
print(unique_chars)
#total number of unique characters (vocab size)

92


In [8]:
for x in index_to_char:
    print(index_to_char[x],x)
    #dictionary mapping


 0
  1
! 2
" 3
# 4
% 5
& 6
' 7
( 8
) 9
* 10
+ 11
, 12
- 13
. 14
/ 15
0 16
1 17
2 18
3 19
4 20
5 21
6 22
7 23
8 24
9 25
: 26
< 27
= 28
> 29
? 30
@ 31
A 32
B 33
C 34
D 35
E 36
F 37
G 38
H 39
I 40
J 41
K 42
L 43
M 44
N 45
O 46
P 47
Q 48
R 49
S 50
T 51
U 52
V 53
W 54
X 55
Y 56
Z 57
[ 58
\ 59
] 60
^ 61
_ 62
a 63
b 64
c 65
d 66
e 67
f 68
g 69
h 70
i 71
j 72
k 73
l 74
m 75
n 76
o 77
p 78
q 79
r 80
s 81
t 82
u 83
v 84
w 85
x 86
y 87
z 88
{ 89
| 90
} 91


In [0]:
all_chars = np.asarray([char_to_index[c] for c in data], dtype = np.int32)

In [10]:
print(len(all_chars))

175731


In [13]:
sequences = list()
for i in range(SEQ_LEN, len(all_chars)):
    # select sequence of tokens
    seq = all_chars[i-SEQ_LEN:i+1]
    sequences.append(seq)
print('Total Sequences: %d' % len(sequences))


Total Sequences: 175667


In [0]:
sequences = np.asarray(sequences)
X, y = sequences[:,:-1], sequences[:,-1]
# X - sequence of 64 characters
# y - next character

In [15]:
for i in X[0]:
    print(index_to_char[i], end='')


X: 1
T:Aunt Hessie's White Horse
% Nottingham Music Database
S:

In [16]:
print(index_to_char[y[0]])

K


In [17]:
for i in sequences[0]:
    print(index_to_char[i], end='')


X: 1
T:Aunt Hessie's White Horse
% Nottingham Music Database
S:K

In [0]:
seq = [to_categorical(x, num_classes=unique_chars) for x in X]
X = np.asarray(seq)
y = to_categorical(y, num_classes=unique_chars)
#one hot encoding

**MODEL DEFINITION**

In [19]:
model = Sequential()
model.add(LSTM(256, return_sequences = True, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.3))
model.add(LSTM(128, return_sequences = False))
model.add(Dropout(0.3))
model.add(Dense(unique_chars, activation='softmax'))
model.summary()

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
lstm_1 (LSTM)                (None, 64, 256)           357376    
_________________________________________________________________
dropout_1 (Dropout)          (None, 64, 256)           0         
_________________________________________________________________
lstm_2 (LSTM)                (None, 128)               197120    
_________________________________________________________________
dropout_2 (Dropout)          (None, 128)               0         
_________________________________________________________________
dense_1 (Dense)              (None, 92)                11868     
Total params: 566,364
Trainable params: 566,364
Non-trainable params: 0
_______________

In [0]:
model.load_weights("weights.best.h5")

In [0]:
model.compile(loss = "categorical_crossentropy", optimizer = "adam", metrics = ["accuracy"])

In [0]:
filepath="weights.best.h5"
checkpoint = ModelCheckpoint(filepath, monitor='acc', verbose=1, save_best_only=True, mode='max')
callbacks_list = [checkpoint]

In [32]:
model.fit(X, y, shuffle=True, epochs=10, batch_size=256, callbacks=callbacks_list)

Epoch 1/10
175667/175667 [==============================] - 122s 697us/step - loss: 0.8293 - acc: 0.7361

Epoch 00001: acc improved from 0.73075 to 0.73605, saving model to weights.best.h5
Epoch 2/10
175667/175667 [==============================] - 123s 699us/step - loss: 0.8177 - acc: 0.7398

Epoch 00002: acc improved from 0.73605 to 0.73975, saving model to weights.best.h5
Epoch 3/10
175667/175667 [==============================] - 122s 695us/step - loss: 0.8012 - acc: 0.7448

Epoch 00003: acc improved from 0.73975 to 0.74482, saving model to weights.best.h5
Epoch 4/10
175667/175667 [==============================] - 122s 697us/step - loss: 0.7889 - acc: 0.7487

Epoch 00004: acc improved from 0.74482 to 0.74870, saving model to weights.best.h5
Epoch 5/10
175667/175667 [==============================] - 122s 694us/step - loss: 0.7752 - acc: 0.7532

Epoch 00005: acc improved from 0.74870 to 0.75316, saving model to weights.best.h5
Epoch 6/10
175667/175667 [=============================

**SEQUENCE GENERATION FROM MODEL**

In [0]:
r = random.randint(0, 91)
#random character chosen to start input string
st = index_to_char[r]
start = data.find(st)
input_seq = all_chars[start : start + 64]
#random input string

In [0]:
#GENERATING OUPUT SEQUENCE

pred_output = list()
for i in range(500):
  pred_input = to_categorical(input_seq, num_classes=unique_chars)
  pred_input = np.reshape(pred_input, (1,SEQ_LEN,unique_chars))
  #reshaped to be accepted by model (1,64,92) 
  prediction = model.predict(pred_input, verbose=0)
  index = np.argmax(prediction)
  pred_output.append(index)  
  input_seq = np.append(input_seq, index)  
  #model prediction appended to sequence
  input_seq = input_seq[1:len(input_seq)]  
  #first character from string removed to pass again through model

In [87]:
for i in pred_output:
  print(index_to_char[i], end='')

4
L:1/8
R:Hornpipe
K:G
P:A
B2|"G"GGBA BdBd|"C"c2ec "D"d2cd|"Am"edcB "D7"A2c2|"G"GBGB "D7"A2A2|
"G"G2G2 G2B2|"C"c2e2 e2g2|"D7"f2e2 d2d2|"G"G2B2 B2G2|"D7"d2f2 d2d2|
"G"d2B2 B2G2|"D7"d2f2 d2d2|"G"B2d2 B2G2|"D7"d2f2 d2d2|"G"G2B2 B2G2|\
"D7"A2A2 A2d2|
"D7"d2c2 A2A2|"D7"d2c2 B2A2|"D7"d2c2 A2F2|"G"G2B2 B2G2|"D7"A2c2 A2A2|
"G"G2G2 B2G2|"D"A2A2 A2F2|"D"A2A2 A2d2|"D"d2f2 d2d2|"D"d2f2 d2d2|
"D"A2A2 d2d2|"D"A2A2 d2d2|"D"A2A2 d2d2|"D"A2A2 d2d2|"D"A2A2 d2d2|
"D"A2A2 d2d2|"D"A2A2 d2d2|"D"A2A2 d2d2|"D"A2A2 d2d2